# Pengujian Model Serving dengan REST API (TensorFlow Serving)
Notebook ini melakukan validasi inferensi (*prediction request*) terhadap model harga rumah yang dideploy menggunakan TensorFlow Serving pada port 8501.

### Cek Status Model

In [2]:
import json
import requests

status_url = 'http://localhost:8501/v1/models/house_pricing_model'
response = requests.get(status_url)
print("Model Status Response:")
print(json.dumps(response.json(), indent=2))

Model Status Response:
{
  "model_version_status": [
    {
      "version": "1",
      "state": "AVAILABLE",
      "status": {
        "error_code": "OK",
        "error_message": ""
      }
    }
  ]
}


## Request Prediksi Menggunakan TFRecord / Raw Data
Karena model diekspor dengan *serving signature* (`serve_tf_examples_fn`), data input dikirim dalam format serialisasi `tf.train.Example` yang di-encode ke Base64 (atau format raw dictionary yang didukung).

### Kirim Sampel Prediksi

In [5]:
import base64
import tensorflow as tf

sample_data = {
    'bathrooms': [2.25],
    'bedrooms': [3.0],
    'condition': [3],
    'floors': [1.5],
    'sqft_above': [1800],
    'sqft_basement': [0],
    'sqft_living': [1800],
    'sqft_lot': [7500],
    'view': [0],
    'waterfront': [0],
    'yr_built': [1975],
    'yr_renovated': [0],
    'city': ['Seattle'],
    'statezip': ['WA 98103'],
}

In [11]:
def create_serialized_example(sample_dict):
  feature = {}
  for key, value in sample_dict.items():
    val = value[0]
    if isinstance(val, float):
      feature[key] = tf.train.Feature(
          float_list=tf.train.FloatList(value=[val])
      )
    elif isinstance(val, int):
      feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[val]))
    elif isinstance(val, str):
      feature[key] = tf.train.Feature(
          bytes_list=tf.train.BytesList(value=[val.encode()])
      )

  example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
  return example_proto.SerializeToString()

In [12]:
serialized_ex = create_serialized_example(sample_data)
b64_ex = base64.b64encode(serialized_ex).decode('utf-8')

Kirim request POST inferensi ke TF Serving

In [17]:
predict_url = (
    "http://localhost:8501/v1/models/house_pricing_model:predict"
)
payload = json.dumps({"instances": [{"b64": b64_ex}]})

headers = {"content-type": "application/json"}
pred_response = requests.post(predict_url, data=payload, headers=headers)

In [18]:
print("Hasil Prediksi Harga Rumah:")
pred_response.json()

Hasil Prediksi Harga Rumah:


{'predictions': [[87451.0234]]}